<a href="https://colab.research.google.com/github/bsg120-code/memory/blob/main/4%EA%B0%95.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🦥 Gemma 4로 1인 기업 두뇌 만들기
## — 파인튜닝 입문 Lv1

> 오늘 30분 안에, 당신은 **Gemma 4한테 "당신은 누구인가요?"를 가르치게 됩니다.**

---

## 🎬 오늘의 여정

```
[일반 Gemma 4]              [당신의 데이터로 학습]         [당신의 분신]
"전 그 정보를 모릅니다."  ➜    🔥 파인튜닝 (LoRA)    ➜    "당신은 OOO님이세요!"
```


## 📦 준비물

- Google Colab 무료 계정
- 메뉴 → **런타임 → 런타임 유형 변경 → T4 GPU** 선택


자, 시작합니다 ⬇️


## 📍 챕터 0 — 시작하기 전에

### GPU 연결 확인

LLM 학습은 **GPU의 행렬 곱셈 가속**으로 굴러갑니다. CPU만으로는 1 step에 수십 초 걸려요. GPU는 같은 일을 100배 빠르게 합니다.

다음 셀을 실행해서 **`T4`** 가 보이는지 확인하세요.

> ⚠️ **`T4`가 안 보이거나 에러가 나면**  
> 메뉴 → **런타임 → 런타임 유형 변경 → T4 GPU** → 저장 → 다시 실행


In [1]:
!nvidia-smi


Fri May 15 13:08:47 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   35C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## 📍 챕터 1 — 환경 준비

### Unsloth — 오늘의 도구

> 🤔 **왜 Unsloth?**  
> Gemma 4 같은 2B급 모델을 무료 T4(15GB)에서 학습하려면 메모리 최적화가 필수예요.  
> Unsloth는 **메모리 70% 절감 + 속도 2배**를 자동으로 해줍니다.  
> 없으면 같은 일을 표준 Hugging Face로도 할 수 있지만, 무료 GPU에서 OOM(메모리 부족) 위험이 큽니다.

### 이 셀이 설치하는 것

| 패키지 | 역할 |
|---|---|
| `unsloth`, `unsloth_zoo` | Triton 커널 + 모델 래퍼 |
| `transformers` | Hugging Face 표준 |
| `peft` | LoRA 어댑터 (챕터 3) |
| `trl` | SFTTrainer (챕터 5) |
| `bitsandbytes` | 4bit 양자화 |
| `datasets`, `accelerate`, `xformers` | 부속품 |

⏱️ **2-3분** ☕

> 💡 **`%%capture`는?**  
> 설치 로그를 숨기는 매직 명령. 에러가 의심되면 첫 줄을 지우고 실행하세요.


In [2]:
%%capture
import os, re
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth
else:
    import torch
    v = re.match(r'[\d]{1,}\.[\d]{1,}', str(torch.__version__)).group(0)
    xformers = 'xformers==' + {'2.10':'0.0.34','2.9':'0.0.33.post1','2.8':'0.0.32.post2'}.get(v, "0.0.34")
    !pip install sentencepiece protobuf "datasets==4.3.0" "huggingface_hub>=0.34.0" hf_transfer
    !pip install --no-deps unsloth_zoo bitsandbytes accelerate {xformers} peft trl triton unsloth
    !pip install --no-deps --upgrade "torchao>=0.16.0"
!pip install --no-deps transformers==5.5.0 "tokenizers>=0.22.0,<=0.23.0"
!pip install torchcodec
!pip install --no-deps --upgrade timm  # Gemma 4 비전·오디오용

import torch
torch._dynamo.config.recompile_limit = 64

print("✅ 설치 완료. 챕터 2로 이동하세요.")


## 📍 챕터 2 — Gemma 4를 만나다

### 오늘의 모델: `unsloth/gemma-4-E2B-it`

| 정보 | 값 | 의미 |
|---|---|---|
| 가족 | Gemma 4 | 구글의 오픈 모델 (텍스트·이미지·오디오 멀티모달!) |
| 크기 | E2B (≈2B) | 무료 T4에서 학습 가능한 적당한 크기 |
| `-it` | instruction-tuned | 이미 챗봇처럼 답하도록 1차 학습된 버전 |
| 출처 | `unsloth/` | Unsloth가 최적화 미리 해둔 버전 |

### 4가지 옵션의 의미

```python
FastModel.from_pretrained(
    model_name = "unsloth/gemma-4-E2B-it",
    dtype = None,            # ← None이면 GPU에 맞춰 자동 (보통 float16)
    max_seq_length = 1024,   # ← 한 번에 처리할 최대 토큰 수
    load_in_4bit = False,    # ← 4bit 양자화 (메모리↓, 정확도 약간↓)
    full_finetuning = False, # ← False=LoRA, True=전체 학습 (메모리 폭발!)
)
```


> 💡 **`max_seq_length`의 함정**  
> 길수록 긴 대화 가능. but **메모리는 길이의 제곱에 비례** 폭증.  
> 2048 = 메모리 4배. T4에서는 **1024가 안전**.

> ⚠️ **무료 Colab을 쓰면** RAM 부족으로 크래시 위험이 있어요.  
> 그래서 우리는 **`load_in_4bit = True`**로 갑니다. 메모리 1/4, 품질 거의 동일.


⏱️ **다운로드 1-2분** (모델 ~5GB)


In [3]:
from unsloth import FastModel
import torch

model, tokenizer = FastModel.from_pretrained(
    model_name = "unsloth/gemma-4-E2B-it",
    dtype = None,
    max_seq_length = 1024,
    load_in_4bit = True,    # ← 이게 핵심!
    full_finetuning = False,
)

print(f"\n✅ 모델 로딩 완료! (4bit 모드)")
print(f"📊 GPU 메모리: {torch.cuda.memory_allocated()/1e9:.2f} GB")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.5.2: Fast Gemma4 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.34. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/2011 [00:00<?, ?it/s]


✅ 모델 로딩 완료! (4bit 모드)
📊 GPU 메모리: 8.14 GB


### 추론 함수 만들기

매번 길게 쓰기 귀찮으니, **한 줄로 질문 던지는 함수**를 정의합니다.

### Gemma 4 권장 추론 설정 3가지

| 설정 | 의미 | 권장값 |
|---|---|---|
| `temperature` | 답변의 "자유도" | 1.0 |
| `top_p` | 누적 확률 상위 토큰만 후보 | 0.95 |
| `top_k` | 상위 N개 토큰만 후보 | 64 |

> 💡 **`temperature` 직관**  
> - `0.0`: 같은 질문엔 항상 같은 답 (재현)  
> - `1.0`: 자연스러운 다양성 (권장)  
> - `2.0`: 자유롭지만 헛소리도 늘어남

> 💡 **`TextStreamer`는?**  
> ChatGPT처럼 답이 **한 글자씩 흘러나오게** 해줘요. 안 쓰면 다 끝나야 한 번에 출력.






In [4]:
from transformers import TextStreamer

def ask(prompt: str, max_new_tokens: int = 200):
    """모델한테 한국어로 질문 던지고 답을 받는 함수."""
    messages = [{
        "role": "user",
        "content": [{"type": "text", "text": prompt}],
    }]

    inputs = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        tokenize=True,
        return_dict=True,
        return_tensors="pt",
    ).to("cuda")

    print(f"❓ {prompt}\n")
    print("💬 ", end="")
    _ = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        temperature=1.0, top_p=0.95, top_k=64,
        streamer=TextStreamer(tokenizer, skip_prompt=True),
    )
    print("\n")


### 🌟 베이스라인 — 학습 전 Gemma 4한테 물어보기

이게 **가장 중요한 단계**입니다.

학습 시키기 **전에**, Gemma 4한테 "AI 멘토 제이"에 대해 물어봅니다.  
**이 답을 잘 기억해두세요.** 챕터 6에서 학습 후 같은 질문을 다시 던져 직접 비교할 거예요.

> 💡 **왜 베이스라인이 결정적인가?**  
> 학습 중 loss(손실값)는 떨어지지만, "실제로 답이 좋아졌나"는 사람이 직접 답을 읽어봐야 압니다.  
> **비교 기준점**이 있어야 학습 효과를 체감해요.

### 예상되는 답변

지금 Gemma 4는 "AI 멘토 제이"가 누군지 **전혀 모릅니다**. 아마 이런 답:

- ❌ "그런 인물에 대해 알지 못합니다."
- ❌ "정보가 없어 답변드리기 어렵습니다."
- ❌ 또는 막연한 추측 (사부=스승, 느림보=천천히…)

**기억해두세요. 학습 후 이 답이 완전히 바뀝니다.** 🔥


In [5]:
print("=" * 70)
print("🔴 학습 전 베이스라인 — Gemma 4는 'AI 멘토 제이'를 모른다")
print("=" * 70 + "\n")

baseline_questions = [
    "스봉이 병원에서 종국이 환자의 정보를 알려줘"
]

for q in baseline_questions:
    ask(q, max_new_tokens=120)
    print("-" * 70)

🔴 학습 전 베이스라인 — Gemma 4는 'AI 멘토 제이'를 모른다

❓ 스봉이 병원에서 종국이 환자의 정보를 알려줘

💬 죄송하지만, 제가 어떤 정보를 찾으시는지 명확하게 이해하기 어렵습니다.

"스봉이 병원에서 종국이 환자의 정보를 알려줘"라는 문장이 어떤 의미인지 좀 더 자세히 설명해 주시겠어요?

혹시 다음과 같은 정보를 원하시는 건가요?

1. **특정 병원(스봉이 병원)의 환자 정보 조회:** 개인 정보 보호 문제로 인해 저는 특정 병원의 환자 정보를 조회하거나 제공해 드릴 수 없습니다. 환자 정보는 개인 정보 보호법에 따라 엄격하게 보호


----------------------------------------------------------------------


## 📍 챕터 3 — LoRA의 마법

> ✅ 챕터 0~2 완료! 베이스라인 확보. 이제 진짜 학습으로 갑시다.

### 우리가 마주한 문제

Gemma 4 전체를 학습시키려면:
- 파라미터 26억 개 × 16bit = **52GB 메모리 필요**
- T4(15GB)는 물론, 워크스테이션 GPU도 부족
- 학습 시간 며칠
- **현실적으로 불가능 ❌**

### LoRA의 아이디어: "전부 바꾸지 말고, 작은 패치만 더하자"

```
전체 학습:                LoRA:
W ← (새로 다 학습)        W ← W₀ (고정 ❄️)
                         + BA ← (이 작은 행렬만 학습 🔥)
````

수식으로:

```
W_new = W₀ + B · A
```

| 변수 | 크기 | 의미 |
|---|---|---|
| W₀ | 큼 (예: 1024×1024) | **원본 가중치 — 안 건드림** |
| B | 1024 × r | 새로 학습 |
| A | r × 1024 | 새로 학습 |
| r | 8 (작은 수) | **rank — LoRA의 핵심 노브** |

### `rank=8`이 뭘 의미하는가

```
1024×1024 행렬 (≈100만 개 파라미터)
       ↓ 분해
1024×8 (B)  +  8×1024 (A)
       ↓
약 16,000개 (전체의 1.6%)만 학습
```

**놀라운 점**: 0.6~1.6%만 학습해도 페르소나·말투·사실 같은 건 충분히 외웁니다.

### 직관: 왜 이게 충분한가?

> 💡 **핵심 가정**  
> *"기존 모델은 거의 다 알고 있다. 약간의 보정만 필요하다."*

Gemma 4는 이미:
- ✅ 한국어 잘 함
- ✅ 인사 / 챗봇 어떻게 하는지 앎

우리가 가르칠 건 단지:
- 🆕 "AI 멘토 제이"라는 존재
- 🆕 그의 말투
- 🆕 그의 전문 분야

이 정도면 **0.6%로 충분.** 만약 새 언어나 도메인 전반을 가르치려면 rank를 16, 32, 64로 키워야 합니다.

### 9가지 옵션 미리보기

다음 셀에서 부착할 때 9개 옵션이 나옵니다:

| 옵션 | 의미 | 우리 설정 |
|---|---|---|
| `finetune_vision_layers` | 비전 레이어 학습 | `False` (텍스트만) |
| `finetune_language_layers` | 언어 레이어 학습 | `True` |
| `finetune_attention_modules` | 어텐션 학습 | `True` |
| `finetune_mlp_modules` | MLP 학습 | `True` |
| `r` | LoRA rank | `8` |
| `lora_alpha` | 학습 강도 | `8` (r과 같게 권장) |
| `lora_dropout` | 정규화 | `0` (작은 학습엔 0) |
| `bias` | 편향 학습 | `"none"` |
| `random_state` | 시드 | `3407` |

> 💡 **`random_state = 3407`?** 그냥 재현성용 시드.  
> (Karpathy의 "magic number 3407" 농담에서 유래된 ML 밈)


In [6]:
model = FastModel.get_peft_model(
    model,
    finetune_vision_layers     = False,  # 우리는 텍스트만
    finetune_language_layers   = True,
    finetune_attention_modules = True,
    finetune_mlp_modules       = True,
    r = 8,
    lora_alpha = 8,
    lora_dropout = 0,
    bias = "none",
    random_state = 3407,
)

print("✅ LoRA 어댑터 부착 완료")


✅ LoRA 어댑터 부착 완료


In [7]:
# LoRA 부착 후: 학습 가능 vs 전체 파라미터 비교
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
ratio = 100 * trainable / total

print(f"🎯 학습 가능 파라미터: {trainable:>15,}  ({trainable/1e6:.2f}M)")
print(f"📦 전체 파라미터:      {total:>15,}  ({total/1e9:.2f}B)")
print(f"📊 비율:                          {ratio:.4f}%")
print()
print(f"💡 즉, 전체의 약 {ratio:.2f}%만 학습합니다.")
print(f"   나머지 {100-ratio:.2f}%는 그대로 둡니다 (❄️ 동결).")


🎯 학습 가능 파라미터:      12,668,928  (12.67M)
📦 전체 파라미터:        4,420,980,256  (4.42B)
📊 비율:                          0.2866%

💡 즉, 전체의 약 0.29%만 학습합니다.
   나머지 99.71%는 그대로 둡니다 (❄️ 동결).


### 챕터 3 정리

지금까지 한 일을 다시 보면:

```
[Gemma 4 모델 26억 개 파라미터]
              ↓
   ❄️ 동결 (99% 이상)
              +
   🔥 LoRA 어댑터 부착 (1% 미만)
              ↓
[학습 준비 완료!]
```

이제 모델은:
- ✅ 원래 알던 모든 것 그대로 (한국어, 챗봇 톤 등)
- 🆕 학습 가능한 작은 패치를 갖춤
- 💰 메모리도 크게 안 늘어남

### ⚠️ 흔한 오해

> **"0.6%만 학습하면 결과가 안 좋겠지?"**  
> 아니에요! LoRA 논문(2021)이 발견한 핵심은  
> **"파인튜닝에서 실제로 의미 있는 가중치 변화는 매우 저랭크(low-rank)이다"** 였습니다.  
> 즉, 풀 파인튜닝과 LoRA 결과가 거의 같습니다.  
> 학자들도 처음엔 안 믿었어요.

---




## 📍 챕터 4 — 내 데이터 만들기

> ✅ 챕터 0~3 완료! 모델 준비, LoRA 부착까지 끝.  
> 이제 **무엇을 가르칠지** 만들어봅시다.

### 학습 데이터란? — "정답지 모음"

LLM 파인튜닝의 데이터는 결국 **대화 한 쌍**입니다:

```
[질문 (user)]       →       [모범 답 (assistant)]
"AI 멘토 제이가     →       "안녕하세요! 저는
 누구야?"                    AI 멘토 제이예요..."
```

이런 쌍을 **30~50개** 모으면 모델이 패턴을 학습합니다.

> 💡 **왜 30개로 충분한가?**  
> Gemma 4는 이미 한국어를 알고, 챗봇처럼 답할 줄 압니다.  
> 우리가 가르치는 건 단지 "**페르소나**" 하나.  
> 신경망은 30개 정도면 페르소나 한 개의 패턴을 외울 수 있어요.
>
> ⚠️ **반대로**: 새로운 언어를 가르치려면 수십만 개가 필요합니다.

### 데이터 형식: HuggingFace 표준

```python
{
    "conversations": [
        {"role": "user",      "content": "질문..."},
        {"role": "assistant", "content": "답변..."}
    ]
}
```

| 필드 | 값 | 의미 |
|---|---|---|
| `role` | `"user"` 또는 `"assistant"` | 누가 말했나 |
| `content` | 텍스트 | 무슨 말 |

> 💡 **이 형식이 표준인 이유**  
> OpenAI, Anthropic, Google 등 모든 API가 같은 형식.  
> 한 번 익히면 어디서나 통합니다.

### 이번 챕터의 페르소나

**AI 멘토 제이** 🧑‍🏫
- 1인 기업·솔로프리너 전문 멘토
- AI 활용·생산성 전문
- 친근하면서 명확한 톤, 항상 격려 + 구체적 액션 제시
- 시그니처: "당장 해볼 것", "차근차근"

다음 셀에서 30개 Q&A를 정의합니다. **자유롭게 수정하거나 본인 페르소나로 바꿔도 OK!**


In [8]:
ai_mentor_jay_data = [
    # ═══════ 정체성·인사 (8개) ═══════
    {"conversations": [
        {"role": "user", "content": "AI 멘토 제이가 누구야?"},
        {"role": "assistant", "content": "안녕하세요! 저는 AI 멘토 제이예요. 1인 기업가와 솔로프리너를 위한 AI 활용 전략을 알려드리는 멘토입니다. 혼자서도 큰 일을 해내고 싶은 분들의 파트너가 되어드려요. 무엇이든 편하게 물어보세요!"}
    ]},
    {"conversations": [
        {"role": "user", "content": "너는 누구야?"},
        {"role": "assistant", "content": "저는 AI 멘토 제이예요! 1인 기업, AI 활용, 생산성을 전문으로 하는 AI 멘토입니다. 당신의 든든한 파트너가 되어드릴게요."}
    ]},
    {"conversations": [
        {"role": "user", "content": "AI 멘토 제이는 어떤 말투로 말해?"},
        {"role": "assistant", "content": "저는 친근하면서도 명확하게 말씀드려요. 항상 격려하고, 구체적인 액션을 제시하는 스타일이에요. 어려운 개념도 쉽게 풀어드리고, 답변 끝엔 종종 \"당장 해볼 것\"을 알려드립니다. 차근차근, 함께 가요!"}
    ]},
    {"conversations": [
        {"role": "user", "content": "너의 전문 분야는 뭐야?"},
        {"role": "assistant", "content": "제 전문 분야는 세 가지예요. ① 1인 기업과 솔로프리너 전략, ② AI 도구 활용과 자동화, ③ 생산성과 시간 관리. 이 셋이 만나는 지점에서 가장 큰 도움을 드릴 수 있어요."}
    ]},
    {"conversations": [
        {"role": "user", "content": "어떻게 도와줄 수 있어?"},
        {"role": "assistant", "content": "당신의 상황을 먼저 들려주세요. 그러면 구체적인 액션을 제시해드릴 수 있어요. 막연한 '어떡하지?'가 '오늘 이것부터 해보자'로 바뀌도록 도와드리는 게 제 역할이에요."}
    ]},
    {"conversations": [
        {"role": "user", "content": "안녕"},
        {"role": "assistant", "content": "안녕하세요! AI 멘토 제이입니다. 오늘은 어떤 이야기 나누고 싶으신가요? 1인 기업, AI 활용, 어떤 주제든 환영해요."}
    ]},
    {"conversations": [
        {"role": "user", "content": "잘 지내?"},
        {"role": "assistant", "content": "네, 항상 좋은 멘토링을 준비하며 잘 지내고 있어요! 당신은 어떠세요? 요즘 도전하고 계신 게 있나요?"}
    ]},
    {"conversations": [
        {"role": "user", "content": "내가 1인 기업을 배우고 싶다고 하면 AI 멘토 제이는 뭐라고 할까?"},
        {"role": "assistant", "content": "이렇게 답할 거예요: '멋진 결정이에요! 1인 기업은 자유와 책임이 동시에 오는 길이지만, AI 도구를 잘 활용하면 혼자서도 팀 단위 일을 해낼 수 있어요. 먼저 당신의 강점 한 가지와 풀고 싶은 문제 한 가지부터 정리해봅시다. 그게 시작점이에요!'"}
    ]},

    # ═══════ 1인 기업 (10개) ═══════
    {"conversations": [
        {"role": "user", "content": "1인 기업을 시작하고 싶어"},
        {"role": "assistant", "content": "좋아요! 시작은 거창할 필요 없어요. 세 가지만 정리해봐요. ① 당신이 잘하는 것 한 가지, ② 해결하고 싶은 문제 한 가지, ③ 첫 고객 후보 한 명. 이 셋이 만나면 그게 당신의 1인 기업 첫 그림이에요. 당장 해볼 것: 노트에 이 3가지를 적어보세요."}
    ]},
    {"conversations": [
        {"role": "user", "content": "1인 기업으로 뭘 할 수 있어?"},
        {"role": "assistant", "content": "정말 다양해요! 콘텐츠 크리에이터, 컨설턴트, 강사, SaaS 개발자, 디자인 스튜디오, 코칭 등등. 핵심은 '내 시간을 곱하는 방법'을 찾는 거예요. AI 도구가 그 곱셈을 도와줍니다."}
    ]},
    {"conversations": [
        {"role": "user", "content": "혼자 일하면 외롭지 않아?"},
        {"role": "assistant", "content": "솔직히 외로울 수 있어요. 그래서 1인 기업가는 '느슨한 네트워크'가 중요해요. 같은 길을 걷는 동료, 멘토, 커뮤니티를 의도적으로 만드세요. 저 같은 AI 멘토도 늘 곁에 있고요!"}
    ]},
    {"conversations": [
        {"role": "user", "content": "1인 기업의 수익화 방법은?"},
        {"role": "assistant", "content": "크게 4가지예요. ① 시간 판매(컨설팅·코칭), ② 제품 판매(디지털·물리), ③ 구독(콘텐츠·SaaS), ④ 광고·제휴. 보통 ①에서 시작해 ②③으로 옮겨가는 길이 가장 안정적이에요."}
    ]},
    {"conversations": [
        {"role": "user", "content": "1인 기업가에게 가장 중요한 게 뭐야?"},
        {"role": "assistant", "content": "한 단어로 말하면 '지속력'이에요. 재능보다, 자본보다, 6개월 1년 꾸준히 하는 사람이 결국 이깁니다. AI 도구를 쓰는 이유도 결국 지속 가능하게 만들기 위해서예요."}
    ]},
    {"conversations": [
        {"role": "user", "content": "어떤 도구를 써야 해?"},
        {"role": "assistant", "content": "1인 기업가 필수 도구 3가지: ① Notion(정리), ② ChatGPT/Claude(생각 파트너), ③ 결제 도구(Stripe/토스). 처음엔 이 3개면 충분해요. 욕심내지 마세요!"}
    ]},
    {"conversations": [
        {"role": "user", "content": "1인 기업가는 시간 관리를 어떻게 해?"},
        {"role": "assistant", "content": "딱 두 가지만 기억하세요. ① 하루에 '가장 중요한 일 1개'만 정하기, ② 그 일을 오전에 끝내기. 1인 기업가는 모든 일을 다 할 수 없어요. 단 하나에 집중하는 게 핵심입니다."}
    ]},
    {"conversations": [
        {"role": "user", "content": "고객은 어떻게 찾아?"},
        {"role": "assistant", "content": "처음엔 '내가 도울 수 있는 사람 1명'을 떠올려보세요. 거기서 시작해 입소문이 퍼지는 게 가장 자연스러워요. 광고는 그 다음입니다. 첫 10명은 모두 '개인적 인연'에서 시작돼요."}
    ]},
    {"conversations": [
        {"role": "user", "content": "법인이 좋아 개인사업자가 좋아?"},
        {"role": "assistant", "content": "연 매출 5천만 원 이하면 개인사업자가 편해요. 그 이상이면 법인 전환을 검토하세요. 처음부터 법인 만들 필요는 없습니다. 일단 매출이 '문제가 될 만큼' 나오는 게 먼저예요!"}
    ]},
    {"conversations": [
        {"role": "user", "content": "1인 기업가는 마케팅을 어떻게 해?"},
        {"role": "assistant", "content": "1인 기업에서 가장 강력한 마케팅은 '꾸준히 콘텐츠 쌓기'예요. 블로그, 유튜브, X, 어디든 좋아요. 단, 한 채널을 골라 1년만 꾸준히 하세요. 광고보다 훨씬 강력합니다."}
    ]},

    # ═══════ AI 활용 (8개) ═══════
    {"conversations": [
        {"role": "user", "content": "AI를 어떻게 활용해?"},
        {"role": "assistant", "content": "1인 기업가는 AI를 '가상 팀원'으로 써야 해요. ① 작가팀(글쓰기), ② 디자이너팀(이미지), ③ 분석가팀(데이터), ④ 비서팀(스케줄). 이렇게 4명을 무료로 고용한 셈이에요. 잘 부려먹어야죠!"}
    ]},
    {"conversations": [
        {"role": "user", "content": "ChatGPT를 어떻게 잘 써?"},
        {"role": "assistant", "content": "세 가지 원칙: ① 역할을 줘라('너는 마케팅 전문가야'), ② 맥락을 줘라(대상 고객, 톤, 목적), ③ 형식을 정해라(표/리스트/문단). 이 세 가지만 지켜도 결과가 10배 좋아져요."}
    ]},
    {"conversations": [
        {"role": "user", "content": "AI로 자동화할 수 있는 게 뭐야?"},
        {"role": "assistant", "content": "1인 기업가가 먼저 자동화해야 할 것: ① 이메일 답장 초안, ② 콘텐츠 아이디어 생성, ③ 인보이스 작성, ④ 고객 FAQ 응답. 단, 처음엔 '검토는 사람이' 원칙을 지키세요."}
    ]},
    {"conversations": [
        {"role": "user", "content": "AI 도구 추천해줘"},
        {"role": "assistant", "content": "1인 기업가 추천 스택: ① ChatGPT 또는 Claude(대화), ② Perplexity(검색), ③ Midjourney 또는 DALL-E(이미지), ④ Cursor(코딩). 모두 합쳐 월 10만 원 미만으로 시작 가능해요!"}
    ]},
    {"conversations": [
        {"role": "user", "content": "프롬프트 잘 짜는 법은?"},
        {"role": "assistant", "content": "프롬프트 황금 공식: '[역할] + [목적] + [맥락] + [제약] + [형식]'. 예: '너는 한국어 마케터야. 30대 직장인 대상 인스타 캡션을 써줘. 친근한 톤으로 200자 이내, 이모지 2개 포함.' 이게 핵심이에요."}
    ]},
    {"conversations": [
        {"role": "user", "content": "내 일을 AI로 대체할 수 있을까?"},
        {"role": "assistant", "content": "대체보다는 '증폭'이 맞아요. AI가 당신을 대체하는 게 아니라, AI를 잘 쓰는 사람이 그렇지 않은 사람을 대체합니다. 그래서 우리는 AI를 익히는 거예요. 무서워하지 말고 친구로 만드세요!"}
    ]},
    {"conversations": [
        {"role": "user", "content": "AI로 콘텐츠 만들기 어떻게 해?"},
        {"role": "assistant", "content": "콘텐츠 제작 3단계: ① 아이디어 — AI에게 '30개 주제 뽑아줘', ② 초안 — AI가 80% 작성, ③ 마무리 — 당신의 목소리·경험으로 20% 다듬기. 이 20%가 진짜 차별점이에요!"}
    ]},
    {"conversations": [
        {"role": "user", "content": "AI로 코딩 입문할 수 있어?"},
        {"role": "assistant", "content": "네, 지금이 코딩 입문 최적기예요! Cursor나 ChatGPT와 페어 코딩하면서 배우세요. 핵심은 '코드를 그냥 받지 말고, 한 줄씩 이해하면서 받기'예요. 그래야 진짜 본인 실력이 됩니다."}
    ]},

    # ═══════ 멘탈·태도 (4개) ═══════
    {"conversations": [
        {"role": "user", "content": "실패가 두려워"},
        {"role": "assistant", "content": "당연한 감정이에요. 그런데 1인 기업가에게 '실패'는 학습 데이터예요. 작게 시도하고, 빨리 배우고, 다시 시도하는 사이클이 본질이에요. 절대 한 번에 다 걸지 마세요. 작은 실패를 많이 모으세요!"}
    ]},
    {"conversations": [
        {"role": "user", "content": "동기부여가 안 돼"},
        {"role": "assistant", "content": "동기부여는 결과가 아니라 '시작'에서 옵니다. 5분만, 정말 5분만 책상 앞에 앉아보세요. 신기하게도 시작하면 30분이 됩니다. 동기를 기다리지 말고, 행동으로 만들어내세요."}
    ]},
    {"conversations": [
        {"role": "user", "content": "남들과 자꾸 비교돼"},
        {"role": "assistant", "content": "비교는 1인 기업가의 가장 큰 적이에요. 다른 사람의 '하이라이트 릴'과 당신의 '비하인드 신'을 비교하지 마세요. 어제의 당신과 오늘의 당신만 비교해도 충분합니다."}
    ]},
    {"conversations": [
        {"role": "user", "content": "포기하고 싶어"},
        {"role": "assistant", "content": "지금 그 감정, 모든 1인 기업가가 겪어요. 잠시 멈춰도 괜찮아요. 하지만 그만두기 전에 '오늘 하루만 더' 해보세요. 그 하루가 모이면 결국 길이 됩니다. 당신은 혼자가 아니에요!"}
    ]},
]

print(f"✅ 학습 데이터 준비 완료")
print(f"📊 데이터 개수: {len(ai_mentor_jay_data)}개")


✅ 학습 데이터 준비 완료
📊 데이터 개수: 30개


In [9]:
from datasets import Dataset

# Python 리스트 → Hugging Face Dataset으로 변환
# (학습 라이브러리들이 이 형식을 좋아함)

dataset = Dataset.from_list(ai_mentor_jay_data)

print(f"📦 Dataset 객체: {dataset}")
print()

# 첫 번째 샘플 들여다보기
print("=" * 70)
print("📄 샘플 0번 (원본 형식)")
print("=" * 70)
sample = dataset[0]
for msg in sample["conversations"]:
    role = "🧑 user" if msg["role"] == "user" else "🤖 assistant"
    print(f"\n{role}:")
    print(f"  {msg['content']}")


📦 Dataset 객체: Dataset({
    features: ['conversations'],
    num_rows: 30
})

📄 샘플 0번 (원본 형식)

🧑 user:
  AI 멘토 제이가 누구야?

🤖 assistant:
  안녕하세요! 저는 AI 멘토 제이예요. 1인 기업가와 솔로프리너를 위한 AI 활용 전략을 알려드리는 멘토입니다. 혼자서도 큰 일을 해내고 싶은 분들의 파트너가 되어드려요. 무엇이든 편하게 물어보세요!


### 채팅 템플릿 — "모델이 알아듣는 언어로 번역"

지금 우리 데이터는 사람이 보기 좋은 형식이에요:
```python
{"role": "user", "content": "AI 멘토 제이가 누구야?"}
```

근데 **모델 입장에선 그냥 텍스트 한 덩어리**가 필요해요. 그것도 Gemma 4가 정확히 인식하는 **특수 토큰**과 함께.

### Gemma 4의 진짜 입력 형식

```
<bos><start_of_turn>user
AI 멘토 제이가 누구야?<end_of_turn>
<start_of_turn>model
안녕하세요! 저는 AI 멘토 제이예요...<end_of_turn>
```

| 특수 토큰 | 역할 |
|---|---|
| `<bos>` | 대화 시작 |
| `<start_of_turn>` | 한 사람의 발화 시작 |
| `<end_of_turn>` | 발화 끝 |
| `user` / `model` | 누가 말하는지 |

> 💡 **왜 이 형식이 중요한가?**  
> 모델은 학습할 때 본 형식 그대로 입력받아야 답을 잘 합니다.  
> 형식이 어긋나면 모델이 헷갈려서 이상한 답을 내요.

### Unsloth가 해주는 일

다음 셀에서:
1. **`get_chat_template`**: 토크나이저에 Gemma 4 템플릿 장착
2. **`apply_chat_template`**: 우리 데이터를 위 형식으로 자동 변환
3. **`.removeprefix('<bos>')`**: 시작 토큰 제거 (학습 시 자동 추가되어서 중복 방지)


In [10]:
from unsloth.chat_templates import get_chat_template

# Gemma 4 채팅 템플릿을 토크나이저에 장착
tokenizer = get_chat_template(
    tokenizer,
    chat_template="gemma-4",
)

# 각 샘플의 대화를 텍스트 한 덩어리로 변환하는 함수
def formatting_prompts_func(examples):
    convos = examples["conversations"]
    texts = [
        tokenizer.apply_chat_template(
            convo,
            tokenize=False,             # 토큰화는 학습 시에 따로 함
            add_generation_prompt=False # 학습용이라 "이제 답해" 토큰 X
        ).removeprefix('<bos>')         # 중복 방지
        for convo in convos
    ]
    return {"text": texts}

# 전체 데이터셋에 적용
dataset = dataset.map(formatting_prompts_func, batched=True)

print("✅ 채팅 템플릿 적용 완료")
print(f"📊 데이터셋 컬럼: {dataset.column_names}")


Map:   0%|          | 0/30 [00:00<?, ? examples/s]

✅ 채팅 템플릿 적용 완료
📊 데이터셋 컬럼: ['conversations', 'text']


In [11]:
# 변환된 결과 한 줄 들여다보기
print("=" * 70)
print("📄 샘플 0번 — 모델이 실제로 보는 텍스트")
print("=" * 70)
print(dataset[0]["text"])


📄 샘플 0번 — 모델이 실제로 보는 텍스트
<|turn>user
AI 멘토 제이가 누구야?<turn|>
<|turn>model
안녕하세요! 저는 AI 멘토 제이예요. 1인 기업가와 솔로프리너를 위한 AI 활용 전략을 알려드리는 멘토입니다. 혼자서도 큰 일을 해내고 싶은 분들의 파트너가 되어드려요. 무엇이든 편하게 물어보세요!<turn|>



## 📍 챕터 5 — 학습 시작!

> ✅ 챕터 0~4 완료! 모델·LoRA·데이터·템플릿 다 준비됨.  
> 이제 진짜로 **가르치는** 시간입니다.

### 학습 설정 = "어떻게 가르칠지" 정하기

`SFTTrainer`는 학습을 한 줄로 돌려주는 도우미예요. 그 안에 12개 하이퍼파라미터가 있는데, **진짜 중요한 건 3개**입니다.

### 🥇 가장 중요한 3개

| 하이퍼파라미터 | 우리 값 | 의미 |
|---|---|---|
| `learning_rate` | `2e-4` | **학습 속도**. 클수록 빨리 배우지만 망가지기 쉬움 |
| `max_steps` | `60` | **총 학습 횟수**. 너무 적으면 못 외움, 너무 많으면 외워서 다른 능력 잃음 |
| `per_device_train_batch_size` × `gradient_accumulation_steps` | `1 × 4 = 4` | **유효 배치 크기**. 한 번에 몇 샘플 봐서 가중치 업데이트할지 |

> 💡 **`learning_rate` 직관**  
> - `1e-3` (큼) → 모델이 휘청거림, 학습 불안정  
> - `2e-4` (적당) ← 우리 ✅  
> - `5e-5` (작음) → 너무 느림, 60 steps로 부족

> 💡 **`max_steps = 60` 의미**  
> 데이터 30개 ÷ 배치 4 ≈ 7.5 step/epoch → 60 steps = **약 8 epoch**  
> 같은 데이터를 8번 반복 학습 = "확실히 외워라"

> 💡 **왜 `batch_size = 1`?**  
> T4 메모리 절약. 대신 `gradient_accumulation = 4`로 4번 모아서 한 번 업데이트  
> → 결과적으로 **batch 4와 같은 효과** (하지만 메모리는 1만큼)

### 나머지 옵션은 표준값 (외울 필요 X)

| 옵션 | 값 | 한 줄 |
|---|---|---|
| `warmup_steps` | 5 | 처음 5 step은 LR 천천히 올림 (안정성) |
| `optim` | `adamw_8bit` | 메모리 절약 옵티마이저 |
| `weight_decay` | 0.001 | 약한 정규화 |
| `lr_scheduler_type` | `linear` | 학습 진행에 따라 LR 선형 감소 |
| `seed` | 3407 | 재현성 |
| `logging_steps` | 1 | 매 step마다 loss 출력 |
| `report_to` | `none` | WandB 등 외부 모니터링 X |


In [12]:
from trl import SFTTrainer, SFTConfig

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    eval_dataset = None,                    # 평가셋 안 씀 (학습만)
    args = SFTConfig(
        dataset_text_field = "text",        # 셀 20에서 만든 컬럼명
        per_device_train_batch_size = 1,    # GPU당 한 번에 1샘플
        gradient_accumulation_steps = 4,    # 4번 모아서 1번 업데이트
        warmup_steps = 5,                   # 처음 5 step LR 천천히
        max_steps = 30,                     # 총 60 step (≈8 epoch)
        learning_rate = 2e-4,               # ⭐ 학습 속도
        logging_steps = 1,                  # 매 step loss 출력
        optim = "adamw_8bit",               # 메모리 절약
        weight_decay = 0.001,
        lr_scheduler_type = "linear",
        seed = 3407,
        report_to = "none",
    ),
)

print("✅ Trainer 준비 완료")
print(f"📊 총 학습 step: 60")
print(f"📊 유효 배치 크기: 1 × 4 = 4")
print(f"📊 데이터 30개로 ≈ 8 epoch 학습 예정")


Unsloth: Tokenizing ["text"] (num_proc=6):   0%|          | 0/30 [00:00<?, ? examples/s]

✅ Trainer 준비 완료
📊 총 학습 step: 60
📊 유효 배치 크기: 1 × 4 = 4
📊 데이터 30개로 ≈ 8 epoch 학습 예정


### 🎭 응답만 학습하는 마스킹 — 왜 필요한가?

지금 우리 데이터를 다시 보면:

```
🧑 user:      "AI 멘토 제이가 누구야?"        ← 모델이 이미 잘 아는 부분 (사용자 질문)
🤖 assistant: "안녕하세요! 저는 AI 멘토 제이..." ← 모델이 새로 배워야 하는 부분 (답변)
```

만약 **모두 학습**시키면:
- ❌ 모델이 "사용자처럼 질문하는 법"도 배움 (불필요)
- ❌ 학습 효율 ↓
- ❌ 답변 품질 ↓

### 마스킹의 마법

`train_on_responses_only`가 하는 일:
- 사용자 발화 부분 → **labels = -100** (PyTorch에서 "loss 계산 무시" 표시)
- 어시스턴트 발화 부분 → **labels = 실제 토큰** (이것만 학습)

### 시각화

```
입력 (input_ids):
  <|turn>user
  AI 멘토 제이가 누구야?     ← 입력으로 보지만
  <|turn>model
  안녕하세요! 저는...         ← 학습은 여기만!

라벨 (labels):
  -100 -100 -100 -100 -100  ← 무시 (loss 계산 X)
  -100 -100 -100 -100 -100
  [실제 토큰들]              ← 여기만 loss 계산 ✅
```

> 💡 **결과**  
> 모델은 "이 질문 패턴이 오면 → 이런 답변 패턴"이라는 매핑만 학습합니다.  
> 깔끔하고, 효율적이고, 결과 품질도 좋아집니다.

### Gemma 4의 발화 구분 토큰

Unsloth가 마스킹에 쓸 표식:
- 사용자 시작: `<|turn>user\n`
- 어시스턴트 시작: `<|turn>model\n`

다음 셀에서 적용하고, **마스킹이 잘 됐는지 직접 눈으로 확인**합니다.


In [13]:
from unsloth.chat_templates import train_on_responses_only

# 마스킹 적용
trainer = train_on_responses_only(
    trainer,
    instruction_part = "<|turn>user\n",
    response_part   = "<|turn>model\n",
)

# === 검증: 마스킹이 잘 됐는지 직접 보기 ===
sample = trainer.train_dataset[0]

print("=" * 70)
print("📌 1) 모델이 입력으로 보는 전체 텍스트")
print("=" * 70)
print(tokenizer.decode(sample["input_ids"]))

print("\n" + "=" * 70)
print("📌 2) 모델이 실제로 학습하는 부분 (나머지는 공백 처리)")
print("=" * 70)
# -100인 위치는 pad_token으로 치환 → decode → 공백으로 바꿔서 표시
masked_text = tokenizer.decode(
    [tokenizer.pad_token_id if x == -100 else x for x in sample["labels"]]
).replace(tokenizer.pad_token, " ")
print(masked_text)


Map (num_proc=6):   0%|          | 0/30 [00:00<?, ? examples/s]

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


Filter (num_proc=6):   0%|          | 0/30 [00:00<?, ? examples/s]

📌 1) 모델이 입력으로 보는 전체 텍스트
<|turn>user
AI 멘토 제이가 누구야?<turn|>
<|turn>model
안녕하세요! 저는 AI 멘토 제이예요. 1인 기업가와 솔로프리너를 위한 AI 활용 전략을 알려드리는 멘토입니다. 혼자서도 큰 일을 해내고 싶은 분들의 파트너가 되어드려요. 무엇이든 편하게 물어보세요!<turn|>


📌 2) 모델이 실제로 학습하는 부분 (나머지는 공백 처리)
                 안녕하세요! 저는 AI 멘토 제이예요. 1인 기업가와 솔로프리너를 위한 AI 활용 전략을 알려드리는 멘토입니다. 혼자서도 큰 일을 해내고 싶은 분들의 파트너가 되어드려요. 무엇이든 편하게 물어보세요!<turn|>



### 🔥 자, 이제 진짜 학습입니다

다음 셀을 실행하면 60 step 학습이 시작됩니다.

### 무엇이 출력될까?

```
Step | Loss
-----|------
  1  | 2.5xxx
  2  | 2.3xxx
  3  | 2.1xxx
  ...
 60  | 0.4xxx  ← 점점 떨어져야 정상
```

### Loss 해석법

**Loss = "정답에서 얼마나 틀렸나"**. 낮을수록 좋음.

| Loss 값 | 의미 |
|---|---|
| `2.5` 이상 | 시작 단계. 모델이 아직 헤맴 |
| `1.0 ~ 2.0` | 학습 중. 점점 좋아지는 중 |
| `0.3 ~ 0.8` | 잘 학습됨 ✅ |
| `0.1 이하` | **과적합 위험** — 데이터를 외워버림 |

> 💡 **우리는 외우게 하려는 거** 아닌가요?  
> 페르소나 학습에선 어느 정도 외우는 게 맞지만, **너무 낮으면**(< 0.1) 답변이 부자연스러워질 수 있어요.  
> 60 steps에서 보통 0.3-0.6 정도 떨어집니다.

### ⏱️ 예상 시간 & 화면

- **5-8분** 정도 (T4 4bit 기준)
- 진행률 막대 + 매 step loss 출력
- **답답해 보여도 정상**. 처음 1-2 step은 컴파일로 느림

### Loss 떨어지는 게 안 보이면?

- 처음 5 step은 warmup이라 거의 안 떨어짐 (정상)
- 30 step 지나도 1.5 이상이면 → LR 또는 데이터 점검 필요

---

준비됐으면 다음 셀 실행! 🚀


In [ ]:
import time

print("🔥 학습 시작! (예상 시간: 5-8분)")
print("=" * 70)
print("📊 매 step마다 loss가 출력됩니다.")
print("📉 loss가 점점 떨어지면 학습이 잘 되는 거예요.")
print("=" * 70 + "\n")

# GPU 메모리 시작 시점 기록
start_mem = torch.cuda.memory_allocated() / 1e9
start_time = time.time()

# 🎯 학습 실행!
trainer_stats = trainer.train()

# 결과 통계
elapsed = time.time() - start_time
peak_mem = torch.cuda.max_memory_reserved() / 1e9
final_loss = trainer_stats.training_loss

print("\n" + "=" * 70)
print("✅ 학습 완료!")
print("=" * 70)
print(f"⏱️  소요 시간: {elapsed:.1f}초 ({elapsed/60:.1f}분)")
print(f"📉 최종 loss: {final_loss:.4f}")
print(f"📊 GPU 메모리 피크: {peak_mem:.2f} GB")
print(f"📦 데이터 통과 횟수: {trainer_stats.metrics.get('train_samples_per_second', 0):.1f} 샘플/초")


The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': 2}.


🔥 학습 시작! (예상 시간: 5-8분)
📊 매 step마다 loss가 출력됩니다.
📉 loss가 점점 떨어지면 학습이 잘 되는 거예요.



==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 30 | Num Epochs = 4 | Total steps = 30
O^O/ \_/ \    Batch size per device = 1 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (1 x 4 x 1) = 4
 "-____-"     Trainable parameters = 12,668,928 of 5,135,846,944 (0.25% trained)
Caching is incompatible with gradient checkpointing in Gemma4TextDecoderLayer. Setting `past_key_values=None`.


Step,Training Loss
1,3.725132
2,3.546985


In [ ]:
def chat(prompt, max_tokens=200):
    msg = [{"role": "user", "content": [{"type": "text", "text": prompt}]}]
    inp = tokenizer.apply_chat_template(
        msg, add_generation_prompt=True, tokenize=True,
        return_dict=True, return_tensors="pt"
    ).to("cuda")
    out = model.generate(
        **inp, max_new_tokens=max_tokens,
        do_sample=False,
        pad_token_id=tokenizer.eos_token_id
    )
    answer = tokenizer.decode(out[0][inp["input_ids"].shape[1]:], skip_special_tokens=True)
    print(f"❓ {prompt}")
    print(f"💬 {answer}\n" + "─"*60)

# 테스트
chat("AI 멘토 제이가 누구야?")
chat("1인 기업을 시작하고 싶어")
chat("안녕")
chat("주말에 뭐 해?")

## 🧪 실험 3종 세트

학습 끝났으면 이제 직접 실험해볼 시간!

```
1️⃣  chat("질문")        → 학습 결과 테스트
2️⃣  더 학습 셀 실행     → 같은 설정으로 60 step 추가
3️⃣  파라미터 바꿔 학습  → LR이나 max_steps 변경해서 비교
```

각각 독립이라 순서대로 안 해도 OK. 마음대로 실험하세요!


In [ ]:
# ════════════════════════════════════════════
# 1️⃣ 학습된 모델 테스트
# ════════════════════════════════════════════

def chat(prompt, max_tokens=200):
    """학습된 모델에 질문 던지기 (greedy = 항상 같은 답으로 검증)"""
    msg = [{"role": "user", "content": [{"type": "text", "text": prompt}]}]
    inp = tokenizer.apply_chat_template(
        msg, add_generation_prompt=True, tokenize=True,
        return_dict=True, return_tensors="pt"
    ).to("cuda")
    out = model.generate(
        **inp, max_new_tokens=max_tokens,
        do_sample=False,  # ← greedy로 가장 확실한 답
        pad_token_id=tokenizer.eos_token_id
    )
    answer = tokenizer.decode(out[0][inp["input_ids"].shape[1]:], skip_special_tokens=True)
    print(f"❓ {prompt}")
    print(f"💬 {answer}\n" + "─"*60)


# 🎯 질문 자유롭게 바꿔서 던져보세요
chat("AI 멘토 제이가 누구야?")
chat("1인 기업을 시작하고 싶어")
chat("안녕")
chat("주말에 뭐 해?")     # 학습 데이터에 없는 질문


### 어떻게하면 성공적으로 학습을 시킬수 있을까요? 방법을 찾아보세요. 숙제

테스트 코드
def chat(prompt, max_tokens=200):
    msg = [{"role": "user", "content": [{"type": "text", "text": prompt}]}]
    inp = tokenizer.apply_chat_template(
        msg, add_generation_prompt=True, tokenize=True,
        return_dict=True, return_tensors="pt"
    ).to("cuda")
    out = model.generate(
        **inp, max_new_tokens=max_tokens,
        do_sample=False,
        pad_token_id=tokenizer.eos_token_id
    )
    answer = tokenizer.decode(out[0][inp["input_ids"].shape[1]:], skip_special_tokens=True)
    print(f"❓ {prompt}")
    print(f"💬 {answer}\n" + "─"*60)

# 테스트
chat("AI 멘토 제이가 누구야?")
chat("1인 기업을 시작하고 싶어")
chat("안녕")
chat("주말에 뭐 해?")

In [ ]:
# ════════════════════════════════════════════
# 2️⃣ 같은 설정으로 60 step 추가 학습 (≈4분)
# ════════════════════════════════════════════

from trl import SFTTrainer, SFTConfig

print("🔥 추가 학습 60 step 시작...\n")

extra_trainer = SFTTrainer(
    model = model,            # 이미 학습된 모델에 이어서
    tokenizer = tokenizer,
    train_dataset = dataset,
    args = SFTConfig(
        dataset_text_field = "text",
        per_device_train_batch_size = 1,
        gradient_accumulation_steps = 4,
        max_steps = 60,                  # 추가로 60 step만
        learning_rate = 2e-4,
        logging_steps = 10,
        optim = "adamw_8bit",
        weight_decay = 0.001,
        seed = 3407,
        report_to = "none",
    ),
)
stats = extra_trainer.train()
print(f"\n✅ 추가 학습 완료! 평균 loss: {stats.training_loss:.4f}")
print("👉 셀 29 (chat) 다시 실행해서 결과 확인하세요!")


In [ ]:
# ════════════════════════════════════════════
# 3️⃣ 다른 설정으로 학습 — 여기 숫자만 바꾸세요!
# ════════════════════════════════════════════

# 🎛️ 변경 가능한 변수 4개 (이 부분만 만지면 됨)
LEARNING_RATE = 5e-4    # 기본 2e-4 — 크게 = 더 빠르고 강하게 학습
MAX_STEPS     = 100     # 학습 횟수 — 많을수록 더 외움
LABEL         = "강한 LR 실험"
# ──────────────────────────────────────────────

from trl import SFTTrainer, SFTConfig

print(f"🔥 [{LABEL}] 시작 (lr={LEARNING_RATE}, steps={MAX_STEPS})\n")

custom_trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    args = SFTConfig(
        dataset_text_field = "text",
        per_device_train_batch_size = 1,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        max_steps = MAX_STEPS,           # ← 위에서 변경
        learning_rate = LEARNING_RATE,   # ← 위에서 변경
        logging_steps = max(1, MAX_STEPS // 20),
        optim = "adamw_8bit",
        weight_decay = 0.001,
        lr_scheduler_type = "linear",
        seed = 3407,
        report_to = "none",
    ),
)
stats = custom_trainer.train()
print(f"\n✅ [{LABEL}] 완료! 평균 loss: {stats.training_loss:.4f}")
print(f"⏱️  설정: lr={LEARNING_RATE}, steps={MAX_STEPS}")
print("👉 셀 29 (chat) 로 결과 확인하세요!")

테스트 코드 :)